In [41]:
import os
import cv2
import yaml
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from facenet_pytorch import InceptionResnetV1
from ipynb.fs.full.tracking import run_tracking_pipeline

**LOADING THE YAML FILE**

In [ ]:
with open("/content/config.yaml", "r") as file:
    config = yaml.safe_load(file)

In [43]:
dataset_path     = config["project"]["base_raw_path"]
embedding_config = config["face_embedding"]
target_config    = config["target"]
 
IMAGE_SIZE     = embedding_config["image_size"]
SIM_THRESHOLD  = embedding_config["similarity_threshold"]
MIN_FACE_SIZE  = embedding_config["min_face_size"]
DEVICE         = embedding_config["device"]
TARGET_PATH    = target_config["image_path"]

**HELPER FUCNTIONS**

In [44]:
model = InceptionResnetV1(pretrained="vggface2").eval().to(DEVICE)

In [45]:
def is_valid_crop(crop):
    if crop is None or crop.size == 0:
        return False
    h, w = crop.shape[:2]
    return h >= MIN_FACE_SIZE and w >= MIN_FACE_SIZE

In [46]:
def get_face_crop(frame, bbox):
    h, w = frame.shape[:2]
    x1, y1, x2, y2 = bbox
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(w, x2), min(h, y2)
    if x2 <= x1 or y2 <= y1:
        return None
    return frame[y1:y2, x1:x2]

In [47]:
def get_embedding(crop):
    try:
        rgb    = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
        img    = Image.fromarray(rgb).resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
        tensor = torch.tensor(np.array(img), dtype=torch.float32).permute(2, 0, 1)
        tensor = (tensor / 255.0 - 0.5) / 0.5
        tensor = tensor.unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            emb = model(tensor)
            emb = F.normalize(emb, p=2, dim=1)
        return emb.squeeze(0).cpu().numpy()
    except Exception:
        return None

In [48]:
def get_target_embedding():
    frame = cv2.imread(TARGET_PATH)
    if frame is None:
        raise FileNotFoundError(f"Target image not found: {TARGET_PATH}")
    crop = get_face_crop(frame, [0, 0, frame.shape[1], frame.shape[0]])
    emb  = get_embedding(crop)
    if emb is None:
        raise ValueError("Could not compute target embedding.")
    print("✅ Target embedding computed")
    return emb

In [49]:
def is_target(embedding, target_embedding):
    similarity = float(np.dot(embedding, target_embedding))
    return similarity > SIM_THRESHOLD

**MAIN FUNCTION**

In [50]:
def run_embedding_pipeline(dataset_path=None):
    if dataset_path is None:
        dataset_path = config["project"]["base_raw_path"]
 
    assert os.path.exists(dataset_path), f"Path not found: {dataset_path}"
 
    tracking_results = run_tracking_pipeline(dataset_path)
    target_embedding = get_target_embedding()
 
    embedding_results = []
    frame_files       = sorted(os.listdir(dataset_path))
 
    for frame_data in tracking_results:
        frame_id = frame_data["frame_id"]
        frame    = cv2.imread(os.path.join(dataset_path, frame_files[frame_id]))
 
        if frame is None or frame.size == 0:
            continue
 
        non_target_tracks = []
 
        for track in frame_data["tracks"]:
            crop = get_face_crop(frame, track["bbox"])
 
            if not is_valid_crop(crop):
                continue
 
            embedding = get_embedding(crop)
 
            if embedding is None:
                continue
 
            if is_target(embedding, target_embedding):
                continue
 
            non_target_tracks.append({
                "track_id"  : track["track_id"],
                "bbox"      : track["bbox"],
                "confidence": track["confidence"],
                "embedding" : embedding.tolist(),
                "crop"      : crop
            })
 
        embedding_results.append({
            "frame_id": frame_id,
            "tracks"  : non_target_tracks
        })
 
    print(f"✅ Embedding done: {len(embedding_results)} frames processed")
    return embedding_results

In [51]:
if __name__ == "__main__":
    embedding_results = run_embedding_pipeline()

âœ… Model loaded: {0: 'FACE'}
âœ… Detection done: 2292 frames
âœ… Preparation done: 2292 frames ready for tracker
âœ… Tracking done: 2292 frames
✅ Target embedding computed
✅ Embedding done: 2292 frames processed
